In [9]:
import pandas as pd
import re
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import WordNetLemmatizer, PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [10]:
df = pd.read_csv('/content/trum_tweet_sentiment_analysis.csv')
df

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0.0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0.0
2,Trump protests: LGBTQ rally in New York https:...,1.0
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0.0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0.0
...,...,...
257577,"RT @LindaSuhler: Seriously, people? Let the la...",0.0
257578,RT @passantino: The White House is going to pr...,0.0
257579,RT @AngusRobertson: Good for the Speaker. It w...,1.0
257580,#TrumpWarnsIranianTerrorism Thanks trump for s...,0.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257582 entries, 0 to 257581
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   text       257582 non-null  object 
 1   Sentiment  257581 non-null  float64
dtypes: float64(1), object(1)
memory usage: 3.9+ MB


In [12]:
stop_words = set(stopwords.words('english'))

def lower_order(text):
    return text.lower()

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_emoji(string):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"
                               u"\U0001F300-\U0001F5FF"
                               u"\U0001F680-\U0001F6FF"
                               u"\U0001F1E0-\U0001F1FF"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r' ', string)

def removeunwanted_characters(document):
    document = re.sub("@[A-Za-z0-9_]+", " ", document)
    document = re.sub("#[A-Za-z0-9_]+", "", document)
    document = re.sub("[^0-9A-Za-z ]", "", document)
    document = remove_emoji(document)
    document = document.replace('  ', "")
    return document.strip()

def remove_punct(text):
    tokenizer = RegexpTokenizer(r"\w+")
    return tokenizer.tokenize(' '.join(text))

def remove_stopwords(text_tokens):
    return [token for token in text_tokens if token not in stop_words]

def lemmatization(token_text):
    wordnet = WordNetLemmatizer()
    return [wordnet.lemmatize(token, pos='v') for token in token_text]

def stemming(text):
    porter = PorterStemmer()
    return [porter.stem(word) for word in text]

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    data = lower_order(dataset)
    data = remove_urls(data)
    data = remove_emoji(data)
    data = removeunwanted_characters(data)
    tokens = word_tokenize(data)
    tokens = remove_punct(tokens)
    tokens = remove_stopwords(tokens)
    if rule == "lemmatize":
        tokens = lemmatization(tokens)
    elif rule == "stem":
        tokens = stemming(tokens)
    else:
        print("Pick between lemmatize or stem")
    return " ".join(tokens)

In [13]:
df['cleaned_text'] = df['text'].dropna().apply(text_cleaning_pipeline)
df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,rtnot drain swamp taxpayer dollars trip advert...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hackers rig fm radio station play antitr...
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new yorkbyvia
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi im piers morgan david beckham awful donald ...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,rt tech firm sue buzzfeed publish unverified t...


In [14]:
from sklearn.model_selection import train_test_split

df = df.dropna(subset=['cleaned_text', 'Sentiment'])

X = df['cleaned_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 206064, Test size: 51517


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)

(206064, 5000)


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.97      0.95     33991
         1.0       0.93      0.88      0.90     17526

    accuracy                           0.94     51517
   macro avg       0.93      0.92      0.93     51517
weighted avg       0.94      0.94      0.94     51517

